# Preprocesamiento (scikit-learn) y Modelado (scikit-learn)

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')



In [2]:
datos = pd.read_csv('aceptados.csv')
pd.options.display.float_format = '{:,.2f}'.format
# Creacion variable objetivo
df = datos[datos["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()

df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)

df = df.drop(columns=["loan_status"])

print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")



Filas:    1,345,310
Columnas: 151


## Preprocesamiento (scikit-learn)
En esta seccion se elimanaran variables con demasiados valores faltantes, se identifican y eliminan vairables que generan data leakage, se seleccionan las variables importantes mediante Mutual Information, se codifican las variables categoricas y se dividen los datos en conjuntos de entrenamiento y prueba

### Seleccion de variables 
Con esto se quiere reducir la dimensionalidad del dataset elimando columnas que no aportan informacion util al modelo

#### Eliminar columnas con alto porcentaje de NA



In [3]:
umbral = 0.25
cols_eliminar = df.columns[df.isnull().mean() > umbral]
df = df.drop(columns=cols_eliminar)

print(f"Columnas eliminadas: {len(cols_eliminar)}")
print(list(cols_eliminar))
print(f"Columnas restantes: {df.shape[1]}")

Columnas eliminadas: 58
['member_id', 'desc', 'mths_since_last_delinq', 'mths_since_last_record', 'next_pymnt_d', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_leng

Se eliminaron todas las columnas con mas del 25% de valores nulos. Fueron eliminadas 58 columnas reduciendo el dataset de 151 a 93 columnas.

#### Elimianr variables con data leakage

El data leakage ocurre cuando el modelo recibe informacion que no estaria disponible al momento de hacer una prediccion real


In [4]:
leakage = ["recoveries", "collection_recovery_fee", "total_rec_prncp",
    "total_rec_int", "total_rec_late_fee", "last_pymnt_amnt",
    "last_pymnt_d", "next_pymnt_d", "total_pymnt", "total_pymnt_inv",
    "out_prncp", "out_prncp_inv", "debt_settlement_flag",
    "last_credit_pull_d", "issue_d", "last_fico_range_high",
    "last_fico_range_low"]

df = df.drop(columns=[col for col in leakage if col in df.columns])

print(f"Columnas eliminadas por leakage: {len([col for col in leakage if col in df.columns])}")
print(list)
print(f"Columnas restantes: {df.shape[1]}")

Columnas eliminadas por leakage: 0
<class 'list'>
Columnas restantes: 77


Se eliminaron 16 variables con data leakage, reduciendo el dataset de 93 a 77 columnas

#### Eliminar variables "basura"
Variables como `id`, `url`, `emp_title` y `title` no aportan información útil al modelo. `pyment_plan`y `hardship_flag` no tienen varianza, el modelo con un valor constante no aprende nada

In [5]:
cols_basura = ['id', 'url', 'emp_title', 'title','pymnt_plan', 'hardship_flag']
df = df.drop(columns=[col for col in cols_basura if col in df.columns])

print(f"Columnas eliminadas: {len(cols_basura)}")
print(f"Columnas restantes: {df.shape[1]}")

Columnas eliminadas: 6
Columnas restantes: 71


Se eliminaron las 6 variables basura reduciendo el dataset a 71 columna

### Division train/test
Se dividen los datos en 80% entrenamiento y un 20% prueba de forma estratificada, garantizando asi la proporcion de default se mantenga igual en ambos conjuntos

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Proporción default en train: {y_train.mean():.2%}")
print(f"Proporción default en test:  {y_test.mean():.2%}")

X_train: (1076248, 70)
X_test:  (269062, 70)
Proporción default en train: 19.96%
Proporción default en test:  19.96%


Se confirma que la division estratificada quedo igual

### Imputacion de variables faltantes
Se imputan los valores faltantes con la mediana para variables numéricas y la moda para variables categóricas, calculadas únicamente sobre el conjunto de entrenamiento para evitar data leakage

In [8]:
categoricas_imp = ['grade', 'sub_grade', 'emp_length', 'home_ownership', 
                   'verification_status', 'purpose', 'term', 'initial_list_status', 
                   'application_type', 'disbursement_method', 'addr_state', 
                   'zip_code', 'earliest_cr_line']

numericas_imp = [col for col in X_train.columns if col not in categoricas_imp]

# Imputar numéricas con mediana
for col in numericas_imp:
    mediana = X_train[col].median()
    X_train[col] = X_train[col].fillna(mediana)
    X_test[col]  = X_test[col].fillna(mediana)

# Imputar categóricas con moda
for col in categoricas_imp:
    moda = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(moda)
    X_test[col]  = X_test[col].fillna(moda)

print(f"NaN en X_train tras imputación: {X_train.isnull().sum().sum()}")
print(f"NaN en X_test  tras imputación: {X_test.isnull().sum().sum()}")

NaN en X_train tras imputación: 0
NaN en X_test  tras imputación: 0


Confirmamos que quedan 0 NA tras imputacion

### Codificacion de variables categoricas
Se usaran distintas formas de codificacion para variables categoricas dependiendo si son ordinales, nominales, o si las variables que tienen muchos unicos

In [9]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Cambiiar earliest_cr_line a años desde apertura para que sea mas facil tratarla
X_train['cr_line_years'] = 2026 - pd.to_datetime(X_train['earliest_cr_line'], format='%b-%Y', errors='coerce').dt.year
X_test['cr_line_years']  = 2026 - pd.to_datetime(X_test['earliest_cr_line'],  format='%b-%Y', errors='coerce').dt.year

X_train = X_train.drop(columns=['earliest_cr_line'])
X_test  = X_test.drop(columns=['earliest_cr_line'])

print(f"Columnas: {X_train.shape[1]}")
print(f"Ejemplo cr_line_years: {X_train['cr_line_years'].describe()}")

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
# Ordinal encoder: variables ordinales
ordinal_cols = ['grade', 'sub_grade', 'emp_length']
ordinal_categories = [
    ['A','B','C','D','E','F','G'],
    ['A1','A2','A3','A4','A5','B1','B2','B3','B4','B5',
     'C1','C2','C3','C4','C5','D1','D2','D3','D4','D5',
     'E1','E2','E3','E4','E5','F1','F2','F3','F4','F5',
     'G1','G2','G3','G4','G5'],
    ['< 1 year','1 year','2 years','3 years','4 years','5 years',
     '6 years','7 years','8 years','9 years','10+ years']
]

oe = OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1)
X_train[ordinal_cols] = oe.fit_transform(X_train[ordinal_cols])
X_test[ordinal_cols]  = oe.transform(X_test[ordinal_cols])

# OHE: variables nominales
ohe_cols = ['home_ownership', 'verification_status', 'purpose', 'term',
            'initial_list_status', 'application_type', 'disbursement_method']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_ohe = ohe.fit_transform(X_train[ohe_cols])
X_test_ohe  = ohe.transform(X_test[ohe_cols])

X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(ohe_cols), index=X_train.index)
X_test_ohe  = pd.DataFrame(X_test_ohe,  columns=ohe.get_feature_names_out(ohe_cols), index=X_test.index)

X_train = X_train.drop(columns=ohe_cols).join(X_train_ohe)
X_test  = X_test.drop(columns=ohe_cols).join(X_test_ohe)

# Frequency Encoding: zip_code y addr_state
for col in ['zip_code', 'addr_state']:
    freq_map = X_train[col].value_counts(normalize=True)
    X_train[col + '_freq'] = X_train[col].map(freq_map)
    X_test[col + '_freq']  = X_test[col].map(freq_map).fillna(0)
    X_train = X_train.drop(columns=[col])
    X_test  = X_test.drop(columns=[col])

print(f"Dimensiones X_train: {X_train.shape}")
print(f"Dimensiones X_test:  {X_test.shape}")

Columnas: 70
Ejemplo cr_line_years: count   1,076,248.00
mean           27.30
std             7.60
min            11.00
25%            22.00
50%            26.00
75%            31.00
max            92.00
Name: cr_line_years, dtype: float64
Dimensiones X_train: (1076248, 94)
Dimensiones X_test:  (269062, 94)


Despues de realizar la codificacion de las variables nos quedan con 94 columnas. Adicionalmente convertimos la variable `earliest_cr_line` en años de antiguedad crediticia para que sea mas facil para el modelo aprender de ella

### Escalado
En este apartado se escalan las variables numericas con StandardScaler para que todos tengan media 0 y desviacion estandar 1

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train)
X_test_final  = scaler.transform(X_test)

print(f"Dimensiones finales train: {X_train_final.shape}")
print(f"Dimensiones finales test:  {X_test_final.shape}")

Dimensiones finales train: (1076248, 94)
Dimensiones finales test:  (269062, 94)


## Modelado con scikit-learn


### RandomForestClassifier

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import time

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [5, 10, 15]
}

rf = RandomForestClassifier(random_state=42, class_weight='balanced')

grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

start = time.time()
grid_search.fit(X_train_final, y_train)
end = time.time()

print(f"Mejor combinación: {grid_search.best_params_}")
print(f"Mejor ROC AUC: {grid_search.best_score_:.4f}")
print(f"Tiempo de entrenamiento: {end - start:.2f} segundos")

Fitting 3 folds for each of 9 candidates, totalling 27 fits


KeyboardInterrupt: 

In [ ]:
df_sklearn = df.copy()

### Metricas de evaluacion
A continuacion, se evaluara el desempeño del modelo con las siguientes metricas: Accuracy, Precision, Recall F-1score, Matriz de confusion entre otras.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

start_pred = time.time()
y_pred = grid_search.best_estimator_.predict(X_test_final)
end_pred = time.time()

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_pred):.4f}")
print(f"\nMatriz de confusión:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nTiempo de predicción: {end_pred - start_pred:.2f} segundos")

Accuracy:  0.6378
Precision: 0.3096
Recall:    0.6618
F1-score:  0.4218
ROC AUC:   0.6468

Matriz de confusión:
[[136071  79279]
 [ 18164  35548]]

Tiempo de predicción: 2.60 segundos


El modelo obtuvo un Accuracy de 60.78%, un Recall de 71.40% indicando que detecta 7 de cada 10 defaults reales, y una Precision de 29.84% lo que significa que genera 
bastantes falsas alarmas. El F1-score de 0.42 y ROC AUC de 0.6477 nos muestran un desempeño moderado del modelo.

### Feature Importance
Debido a las limitaciones de memoria con el dataset de 368.878 columnas generadas por el OneHotEncoder, no es posible aplicarle Permutation Importance directamente sobre el modelo entrenado. Como alternativa se utiliza Feauture Importance que tiene mucho menos costo computacional.


In [ ]:
# Nombres de las columnas
nombres = numericas + list(ohe.get_feature_names_out(categoricas))

feature_imp = pd.DataFrame({
    "feature": nombres,
    "importance": grid_search.best_estimator_.feature_importances_
}).sort_values(by="importance", ascending=False)

# Agrupar por variable original
todas_vars = numericas + categoricas

def get_variable_original(feature_name):
    for var in todas_vars:
        if feature_name.startswith(var):
            return var
    return feature_name

feature_imp["variable_original"] = feature_imp["feature"].apply(get_variable_original)
# Se agrupan y se promedia la importancia por variable original
importancia_agrupada = feature_imp.groupby("variable_original")["importance"].mean().sort_values(ascending=False)
top20_vars = importancia_agrupada.head(20)
pd.set_option('display.float_format', '{:.6f}'.format)
print(importancia_agrupada)


variable_original
term                         0.044698
fico_range_high              0.033264
installment                  0.027494
int_rate                     0.025723
grade                        0.021464
funded_amnt_inv              0.019465
funded_amnt                  0.018327
fico_range_low               0.018288
mort_acc                     0.017523
num_actv_rev_tl              0.013838
num_tl_op_past_12m           0.013072
revol_util                   0.010902
total_rev_hi_lim             0.009875
acc_open_past_24mths         0.009387
verification_status          0.009337
percent_bc_gt_75             0.007672
sub_grade                    0.007464
loan_amnt                    0.007430
open_acc                     0.007027
annual_inc                   0.006341
num_sats                     0.005551
inq_last_6mths               0.005010
num_op_rev_tl                0.004615
dti                          0.004325
home_ownership               0.004102
num_bc_tl                    0.0

Los resultados muestran que `int_rate` es la variable más importante con 0.073, 
seguida de `dti` con 0.034 y `grade` con 0.020. Variables importantes como 
`funded_amnt`, `fico_range_high`, `term` y `loan_amnt` tienen importancias similares 
alrededor de 0.017. 

Variables con importancia cercana a 0 como `title`, `emp_length` y `policy_code` 
aportan muy poca información al modelo por lo que se eliminaran y haremos un reentreno

#### Filtracion de variables para reentreno
Con las 20 variables mas importantes filtraremos el dataset

In [ ]:
# Filtrar columnas que contienen las variables top20
top20_cols = [col for col in feature_imp["feature"] 
              if any(col.startswith(var) for var in top20_vars)]

# Índices de esas columnas en X_train_final
indices = [i for i, col in enumerate(nombres) if col in top20_cols]

# Filtrar matrices
X_train_top20 = X_train_final[:, indices]
X_test_top20 = X_test_final[:, indices]

print(f"Columnas seleccionadas: {X_train_top20.shape[1]}")

TypeError: startswith first arg must be str or a tuple of str, not float

De las 367.878 columnas solo conservamos 64 de estas

### Reentrenamiento con variables seleccionadas


In [ ]:
start2 = time.time()
grid_search2 = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search2.fit(X_train_top20, y_train)
end2 = time.time()

print(f"Mejor combinación: {grid_search2.best_params_}")
print(f"Mejor ROC AUC: {grid_search2.best_score_:.4f}")
print(f"Tiempo de entrenamiento: {end2 - start2:.2f} segundos")

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Mejor combinación: {'max_depth': 15, 'n_estimators': 100}
Mejor ROC AUC: 0.7135
Tiempo de entrenamiento: 3719.67 segundos


### Metricas del modelo reentrenado

In [ ]:
start_pred2 = time.time()
y_pred2 = grid_search2.best_estimator_.predict(X_test_top20)
end_pred2 = time.time()

print(f"Accuracy:  {accuracy_score(y_test, y_pred2):.4f}")
print(f"Precision: {precision_score(y_test, y_pred2):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred2):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred2):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_pred2):.4f}")
print(f"\nMatriz de confusión:\n{confusion_matrix(y_test, y_pred2)}")
print(f"\nTiempo de predicción: {end_pred2 - start_pred2:.2f} segundos")

Accuracy:  0.6638
Precision: 0.3256
Recall:    0.6388
F1-score:  0.4314
ROC AUC:   0.6544

Matriz de confusión:
[[144298  71052]
 [ 19403  34309]]

Tiempo de predicción: 3.55 segundos


Del modelo reentrenado con las 20 variables se puede ver una mejora en la mayoria de las metricas. El Accuracy mejoro de 60.78% a 66.38%. La Precision de 29.84% a 32.56% y el f1-score de 0.42 a 0.43. Sin embargo el Recall bajo de 71.40% a 63.88% lo que signfica que el modelo detecta menos default correctamente

### Interpretabilidad con LIME

In [ ]:
import lime
import lime.lime_tabular


# Instancias mal clasificadas
y_pred2 = grid_search2.best_estimator_.predict(X_test_top20)
mal_clasificados = np.where(y_pred2 != y_test.values)[0]

print(f"Total mal clasificados: {len(mal_clasificados)}")
print(f"Primeros índices: {mal_clasificados[:5]}")

NameError: name 'grid_search2' is not defined